In [16]:
# Installing packages
# !pip install polars

In [1]:
# Import Packages
import polars as pl
import polars.selectors as cs
import os
import pyarrow.dataset as pads

# Reading in Data

In [4]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# For the data_path argument, include the full file path to the folder that holds the data!    
def readDataSubset(table_type, data_path="/Users/jamesccoats/Documents/SMT-Data-Challenge-2026"):
    '''
    Takes a the set of tables from Data set and
    transforms into panda data frame for manipulation
    
    Input:
    -     table_type   = Set of tables of interest from Data set [str]
    -     data_path    = File Path which data is located in Local device, change as necessary[str]
    
    '''
    if table_type not in ['ball-positions', 'ball-events', 'player-positions', 'lineups', 'game-info']:
        print("Invalid data subset name. Please try again with a valid data subset.")
        return -1

    if table_type == 'lineups' or table_type == 'game-info':
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [5]:
#######################################################
### Code Taken from SMT_Data_Challenge_Demo
#######################################################

# Filter down to only 1 game for ease of computation
# Note: I went through and selected this game for a reason. When you're running 
# this, you're not going to do it game by game. But we'll talk about that more
# next time
# 
# Whenever we have some type of text that we want a programming languge to recognize,
# we need to put that text in quotes.
selected_game_str = "y1_d061_VKA_PHD"

# Read in the ball-events Data and filter to selected game
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read in the ball_pos Data and filter to selected game
ball_pos_subset = readDataSubset('ball-positions')
ball_pos = ball_pos_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_pos_df = pl.from_pandas(ball_pos)

# Home Run Distance

In [7]:
# First, look for all the homeruns in the dataset
homerun_plays_df = ball_events_df \
                                .filter(pl.col("ball_eventcode") == 11)

# Next, make a list that combines the game_str and play_per_game
homerun_plays = homerun_plays_df["game_string"] + "_" + homerun_plays_df["play_per_game"]

In [9]:
# Next find the time that the ball was hit on those plays

# Add a column that is game_string and play_id together
ball_events_w_game_str_ppg = ball_events_df.with_columns(
                                                    pl.concat_str(
                                                        [
                                                            pl.col("game_string"),
                                                            pl.col("play_per_game")
                                                        ],
                                                        separator = "_"
                                                    ).alias("game_str_ppg")
                                                )

# Filter ball_events_w_game_str_ppg to only HRs
ball_events_hr = ball_events_w_game_str_ppg.filter(
                                                    # Filter to only game_str_ppg in homerun_plays
                                                    # Convert home run play from one column data frame to list
                                                    pl.col("game_str_ppg").is_in(homerun_plays.unique().to_list())
                                                ) \
                                                .filter(
                                                    # Filter to when the ball was hit
                                                    pl.col("ball_eventcode") == 4
                                                ) \
                                                .select( # Select Only Needed Columns
                                                    ["game_string", "play_per_game", "game_str_ppg",
                                                        # Renaming the column "timestamp" as "hit_timestamp"
                                                        pl.col("timestamp").alias("hit_timestamp")
                                                    ]
                                                )

ball_events_hr.head()

game_string,play_per_game,game_str_ppg,hit_timestamp
str,i64,str,i64
"""y1_d061_VKA_PHD""",113,"""y1_d061_VKA_PHD_113""",3617664


In [ ]:
# Find Ball Position throughout the flight of the Homeruns
ball_pos_hr = ball_pos_df.with_columns(
                              # Add a game_str_play_id columns
                               pl.concat_str(
                                [
                                 pl.col("game_string"),
                                 pl.col("play_per_game")
                                ],
                                    separator = "_"
                                ).alias("game_str_ppg")
                            ) .filter(
                               # Filter to only game_str_ppgs in homerun_plays
                               pl.col("game_str_ppg").is_in(homerun_plays.unique().to_list())
                            ) .join( # Join on the game_events_hr to get the timestamp of the hit
                              ball_events_hr,
                              how = "left",
                              on = ["game_string", "play_per_game", "game_str_ppg"]
                            )

In [ ]:
# Get the coordinates of the ball when it was hit
ball_pos_hr_hit = ball_pos_hr \
                              .filter( # Filter to only the point of contact
                                  pl.col("timestamp") == pl.col("hit_timestamp")
                                  .over( # When doing the filter, group by game_str_ppg
                                      "game_str_ppg"
                                  )
                              ) \
                              .select( # Select Needed Columns
                                      ["game_string", "play_per_game", "game_str_ppg",
                                       pl.col("ball_position_x").alias("contact_x"),
                                       pl.col("ball_position_y").alias("contact_y"),
                                       pl.col("ball_position_z").alias("contact_z"),
                                      ]
                              )

In [24]:
# Get the coordinates of the ball when it landed
ball_pos_hr_landed = ball_pos_hr \
                              .filter( # Filter to only the point of contact
                                  pl.col("timestamp") == pl.col("timestamp").max()
                                  .over( # When doing the filter, group by game_str_play_id
                                      "game_str_ppg"
                                  )
                              ) \
                              .select( # Select Needed Columns
                                      ["game_string", "play_per_game", "game_str_ppg",
                                       pl.col("ball_position_x").alias("landed_x"),
                                       pl.col("ball_position_y").alias("landed_y"),
                                       pl.col("ball_position_z").alias("landed_z"),
                                      ]
                              )

In [25]:
# Combine ball_pos_hr_hit and ball_pos_hr_landed
coordinates_final_hr = ball_pos_hr_hit.join(
                              ball_pos_hr_landed,
                              how = "left",
                              on = ["game_string", "play_per_game", "game_str_ppg"]
                          )

In [ ]:
# Calculate HR Distances
distances_hr = coordinates_final_hr \
                                    .with_columns( # Add Column for change in x, y and z
                                        (pl.col("landed_x") - pl.col("contact_x")).alias("change_x"),
                                        (pl.col("landed_y") - pl.col("contact_y")).alias("change_y"),
                                        (pl.col("landed_z") - pl.col("contact_z")).alias("change_z"),
                                    ) \
                                    .with_columns( # Add a column that calculates HR Distance
                                        (pl.col("change_x")**2 + pl.col("change_y")**2 + pl.col("change_z")**2).sqrt().alias("hr_distance")
                                    ) \
                                    .select( # Only keep necessary columns
                                        ["game_str_ppg", "hr_distance"]
                                    )

shape: (1, 2)
┌─────────────────────┬─────────────┐
│ game_str_ppg        ┆ hr_distance │
│ ---                 ┆ ---         │
│ str                 ┆ f64         │
╞═════════════════════╪═════════════╡
│ y1_d061_VKA_PHD_113 ┆ 372.23389   │
└─────────────────────┴─────────────┘


# Launch Angle

In [ ]:
# First, find a play where the ball was hit
ball_events_bip = ball_events_df \
                                 .filter(
                                     (pl.col("player_id") == 10) &
                                     (pl.col("ball_eventcode") == 4)
                                 ) \
                                 .with_columns(
                                     # Add a game_str_pppg column
                                     pl.concat_str(
                                         [
                                             pl.col("game_string"),
                                             pl.col("play_per_game")
                                         ],
                                         separator = "_"
                                     ).alias("game_str_ppg")
                                 ) \
                                 .select( # Select Only Needed Columns
                                     ["game_string", "play_per_game", "game_str_ppg",
                                         # Renaming the column "timestamp" as "hit_timestamp
                                         pl.col("timestamp").alias("hit_timestamp")
                                     ]
                                )

In [28]:
# Filter ball position to relevant data
ball_pos_bip = ball_pos_df \
                           .with_columns(
                                        # Add a game_str_play_id column
                                        pl.concat_str(
                                            [
                                                pl.col("game_string"),
                                                pl.col("play_per_game")
                                            ],
                                            separator = "_"
                                        ).alias("game_str_ppg")

                           ) \
                             .filter( # Then filter to only game_str_play_id where there was a hit
                                 pl.col("game_str_ppg").is_in(ball_events_bip["game_str_ppg"].to_list())
                             )

In [29]:
# Next, we only care about the timestamps after contact
ball_pos_after_contact = ball_pos_bip.join(ball_events_bip,
                                           on = ["game_string", "play_per_game", "game_str_ppg"],
                                           how = "left"
                                          ) \
                                      .filter(
                                          pl.col("timestamp") >= pl.col("hit_timestamp")
                                      )

In [ ]:
# For Launch Angle, we want the first 20 timestamps per hit to create stable metrics
# Code Stolen From:
# https://stackoverflow.com/questions/71273072/how-to-the-get-the-first-n-of-a-group-in-polars
ball_pos_first_20 = ball_pos_after_contact \
                                            .sort( # Order Chronologically
                                                ["game_string", "play_per_game", "timestamp"]
                                            ) \
                                            .group_by( # Group by game_str_ppg
                                                ["game_str_ppg"]
                                            ) \
                                            .agg( # Get the first 20 rows by group for all columns
                                                pl.all().head(20)
                                            ) \
                                            .explode( # Explode back out into multiple rows rather than 1 row per group
                                                pl.exclude("game_str_ppg")
                                            ) \
                                            .select(
                                                [
                                                    "game_string", "play_per_game", "game_str_ppg", "timestamp",
                                                    #' I don't want to write out ball_position_x, ball_position_y, 
                                                    #' ball_position_z because that's a lot of typing. Thankfully,
                                                    #' there's a function in the `polars.selectors` package that is called `starts_with`
                                                    #' that selects all the variables that start with that exact phrase for
                                                    #' me
                                                    cs.starts_with("ball_position")
                                                ]
                                            )

In [ ]:
# Finally, calculate Launch Angle
launch_angles_by_play = ball_pos_first_20 \
                                          .sort( # Order Chronologically
                                              ["game_string", "play_per_game", "timestamp"]
                                          ) \
                                         .with_columns( # Find min and max ball positions for x, y and z per game_str_ppg
                                             min_x = pl.col("ball_position_x").min().over(pl.col("game_str_ppg")),
                                             max_x = pl.col("ball_position_x").max().over(pl.col("game_str_ppg")),
                                             min_y = pl.col("ball_position_y").min().over(pl.col("game_str_ppg")),
                                             max_y = pl.col("ball_position_y").max().over(pl.col("game_str_ppg")),
                                             min_z = pl.col("ball_position_z").min().over(pl.col("game_str_ppg")),
                                             max_z = pl.col("ball_position_z").max().over(pl.col("game_str_ppg"))
                                         ) \
                                        .with_columns( # Find the max - min for x y and z
                                            change_x = pl.col("max_x") - pl.col("min_x"),
                                            change_y = pl.col("max_y") - pl.col("min_y"),
                                            change_z = pl.col("max_z") - pl.col("min_z"),
                                        ) \
                                        .with_columns( # Make a column for ground distance covered
                                            ground_distance = (pl.col("change_x")**2 + pl.col("change_y")**2).sqrt()
                                        ) \
                                        .with_columns( # Calculate Launch Angle
                                            launch_angle_rad = (pl.col("change_z") / pl.col("ground_distance")).arctan(),
                                            launch_angle_deg = (pl.col("change_z") / pl.col("ground_distance")).arctan().degrees()
                                        ) \
                                        .select( # Select Only Needed Columns
                                            [
                                                "game_str_ppg", "launch_angle_deg"
                                            ]
                                        ) \
                                        .unique( # Only Unique Combos
                                        ) \
                                        .with_columns( # Calculate Hit Classification like MLB Does
                                            # Launch Angle < 10 ~ Ground Ball
                                            pl.when(pl.col("launch_angle_deg") < 10)
                                                  .then(pl.lit("Ground ball"))
                                            # Launch Angle < 25 ~ Line Drive
                                              .when(pl.col("launch_angle_deg") < 25)
                                                  .then(pl.lit("Line drive"))
                                            # Launch Angle < 50 ~ Fly Ball
                                              .when(pl.col("launch_angle_deg") < 50)
                                                  .then(pl.lit("Fly ball"))
                                            # Launch Angle > 50 and not NaN ~ Pop Up
                                              .when((pl.col("launch_angle_deg") > 50) &
                                                   (pl.col("launch_angle_deg").is_not_nan()))
                                                  .then(pl.lit("Pop up"))
                                              # Otherwise, we're going to assign a None Value
                                              .otherwise(pl.lit(None))
                                              # We're going to call this column hit_classification
                                              .alias("hit_classification")
                                        ) \
                                        .filter( # Get rid of when hit_classification is NULL (these are Foul Balls)
                                            pl.col("hit_classification").is_not_null()
                                        )

# Exit Velo

In [32]:
# Find the first 2 records per BIP
ball_pos_first_2_after_contact = ball_pos_after_contact \
                                            .sort( # Sort Data Frame Chronologically
                                                ["game_string", "play_per_game", "timestamp"]
                                            )  \
                                            .group_by( # Group by game_str_play_id
                                                ["game_str_ppg"]
                                            ) \
                                            .agg( # Get the first 2 rows by group for all columns
                                                pl.all().head(2)
                                            ) \
                                            .explode( # Explode back out into multiple rows rather than 1 row per group
                                                pl.exclude("game_str_ppg")
                                            ) \
                                            .with_columns( # Label every row as either the first or second per game_str_play_id
                                                row_num = pl.int_range( # Giving the range of numbers to assign
                                                    start = 1,
                                                    end = pl.len() + 1,
                                                    step = 1
                                                ) \
                                                .over( # Assign values in groups (by game_str_play_id)
                                                    ["game_str_ppg"]
                                                )
                                            ) \
                                            .select(
                                                [
                                                    "game_string", "play_per_game", "game_str_ppg", "row_num",
                                                    "timestamp", cs.starts_with("ball_position")
                                                ]
                                            )

In [ ]:
# Seperate the first and second records into seperate data frames then join into 1 wider data.frame
ball_pos_hit_1 = ball_pos_first_2_after_contact \
                                                .filter(
                                                    pl.col("row_num") == 1
                                                ) \
                                                .select(
                                                    [
                                                        "game_string", "play_per_game", "game_str_ppg",
                                                        # Rename ball_position and time columns with _1
                                                        pl.col("ball_position_x").alias("x_1"),
                                                        pl.col("ball_position_y").alias("y_1"),
                                                        pl.col("ball_position_z").alias("z_1"),
                                                        pl.col("timestamp").alias("time_1")
                                                    ]
                                                )

ball_pos_hit_2 = ball_pos_first_2_after_contact \
                                                .filter(
                                                    pl.col("row_num") == 2
                                                ) \
                                                .select(
                                                    [
                                                        "game_string", "play_per_game", "game_str_ppg",
                                                        # Rename ball_position and time columns with _2
                                                        pl.col("ball_position_x").alias("x_2"),
                                                        pl.col("ball_position_y").alias("y_2"),
                                                        pl.col("ball_position_z").alias("z_2"),
                                                        pl.col("timestamp").alias("time_2")
                                                    ]
                                                )

ball_pos_hit = ball_pos_hit_1.join(
    ball_pos_hit_2,
    how = "full",
    on = ["game_string", "play_per_game", "game_str_ppg"],
    # Forces the two game_str_play_ids to combine into 1 column together
    # Stolen From:
    # https://docs.pola.rs/user-guide/transformations/joins/#full-join
    coalesce = True
)

In [34]:
# Calculate Exit Velo
exit_velos = ball_pos_hit \
                          .with_columns( # Calculate the differences in x, y, z, and time
                              change_x = pl.col("x_2") - pl.col("x_1"),
                              change_y = pl.col("y_2") - pl.col("y_1"),
                              change_z = pl.col("z_2") - pl.col("z_1"),
                              change_time = pl.col("time_2") - pl.col("time_1"),
                          ) \
                          .with_columns( # Calculate 3D Distance Traveled
                              dist = (pl.col("change_x")**2 + pl.col("change_y")**2 + pl.col("change_z")**2).sqrt()
                          ) \
                          .with_columns( # Calculate dist / time (ft/ms)
                              exit_velo_ft_ms = pl.col("dist") / pl.col("change_time")
                          ) \
                          .with_columns(
                              exit_velo_mph = pl.col("exit_velo_ft_ms") / 5280 * (1000*60*60)
                          ) \
                          .select(
                              [
                                  "game_string", "play_per_game", "game_str_ppg", "exit_velo_mph"
                              ]
                          )

# Combine Everything to 1 Data Frame

In [ ]:
# Combine HR Distances, Launch Angles, and Exit Velos to 1 DF
bip_metrics = distances_hr \
                            .join( # Join on Launch Angle
                                launch_angles_by_play,
                                how = "full",
                                on = "game_str_ppg",
                                coalesce = True
                            ) \
                            .join( # Join on exit velos
                                exit_velos,
                                how = "full",
                                on = "game_str_ppg",
                                coalesce = True
                            ) \
                            .select( # Select and Order Columns
                                [
                                    "game_string", "play_per_game", "game_str_ppg", "exit_velo_mph",
                                    "launch_angle_deg", "hr_distance", "hit_classification"
                                ]
                            ) \
                            .filter( # Filter out anything where we don't have the metrics for it (Foul Balls)
                                pl.col("exit_velo_mph").is_not_nan()
                            ) \
                            .sort(# Order Rows
                                [
                                    "game_string", "play_per_game"
                                ]
                            )